# TextVectorization Detailed Guide

This notebook explains TextVectorization in practical enterprise terms.

Focus areas:
- Function-based arguments and when to use each
- ML concepts in simple language
- Real-world scenarios and runnable examples

## 1) Setup

In [1]:
import tensorflow as tf

print('TensorFlow:', tf.__version__)

TensorFlow: 2.22.0-dev0+selfbuilt


## 2) Function-Based Arguments and Use Cases

TextVectorization has 3 major layers:
- `standardize`: cleanup and normalization
- `split`: how text becomes tokens
- `output_mode`: final numeric representation

### A) `standardize`

Use built-in for quick baseline.
Use custom external function for enterprise text (IDs, URLs, ticket numbers, emails).

Example benefit:
- `Order#9988 failed on https://ops.example.com`
- becomes `<ORDER_ID> failed on <URL>`

In [2]:
texts = tf.constant([
    'SLA-breach on Order#9988 in EU-West',
    'Payment latency spiked on https://ops.example.com',
    'Ticket TKT-1234 escalated to L2 support'
])

def standardize_light(x):
    x = tf.strings.lower(x)
    x = tf.strings.regex_replace(x, r'[^a-z0-9\- ]', ' ')
    x = tf.strings.regex_replace(x, r'\s+', ' ')
    return tf.strings.strip(x)

def standardize_aggressive(x):
    x = tf.strings.lower(x)
    x = tf.strings.regex_replace(x, r'https?://\S+', ' <URL> ')
    x = tf.strings.regex_replace(x, r'order#?\d+', ' <ORDER_ID> ')
    x = tf.strings.regex_replace(x, r'tkt-\d+', ' <TICKET_ID> ')
    x = tf.strings.regex_replace(x, r'\b\d+\b', ' <NUM> ')
    x = tf.strings.regex_replace(x, r'[^a-z<>_ ]', ' ')
    x = tf.strings.regex_replace(x, r'\s+', ' ')
    return tf.strings.strip(x)

In [3]:
vec_light = tf.keras.layers.TextVectorization(
    standardize=standardize_light,
    split='whitespace',
    output_mode='int',
    output_sequence_length=12
)
vec_aggr = tf.keras.layers.TextVectorization(
    standardize=standardize_aggressive,
    split='whitespace',
    output_mode='int',
    output_sequence_length=12
)
vec_light.adapt(texts)
vec_aggr.adapt(texts)

print('Light vocab:', vec_light.get_vocabulary()[:20])
print('Aggressive vocab:', vec_aggr.get_vocabulary()[:20])

Light vocab: ['', '[UNK]', np.str_('on'), np.str_('to'), np.str_('tkt-1234'), np.str_('ticket'), np.str_('support'), np.str_('spiked'), np.str_('sla-breach'), np.str_('payment'), np.str_('order'), np.str_('ops'), np.str_('latency'), np.str_('l2'), np.str_('in'), np.str_('https'), np.str_('example'), np.str_('eu-west'), np.str_('escalated'), np.str_('com')]
Aggressive vocab: ['', '[UNK]', np.str_('>'), np.str_('<'), np.str_('on'), np.str_('_'), np.str_('west'), np.str_('to'), np.str_('ticket'), np.str_('support'), np.str_('spiked'), np.str_('sla'), np.str_('payment'), np.str_('latency'), np.str_('l'), np.str_('in'), np.str_('eu'), np.str_('escalated'), np.str_('breach')]


### B) `split`

Use whitespace split for normal sentences.
Use custom split for domain tokens such as `sla-breach`, `eu-west`, `error-code-500`.

In [4]:
def split_on_space_and_hyphen(x):
    x = tf.strings.regex_replace(x, '-', ' ')
    return tf.strings.split(x)

vec_default_split = tf.keras.layers.TextVectorization(
    standardize=standardize_light,
    split='whitespace',
    output_mode='int',
    output_sequence_length=12
)
vec_custom_split = tf.keras.layers.TextVectorization(
    standardize=standardize_light,
    split=split_on_space_and_hyphen,
    output_mode='int',
    output_sequence_length=12
)
vec_default_split.adapt(texts)
vec_custom_split.adapt(texts)

print('Default split vocab:', vec_default_split.get_vocabulary()[:20])
print('Custom split vocab:', vec_custom_split.get_vocabulary()[:20])

Default split vocab: ['', '[UNK]', np.str_('on'), np.str_('to'), np.str_('tkt-1234'), np.str_('ticket'), np.str_('support'), np.str_('spiked'), np.str_('sla-breach'), np.str_('payment'), np.str_('order'), np.str_('ops'), np.str_('latency'), np.str_('l2'), np.str_('in'), np.str_('https'), np.str_('example'), np.str_('eu-west'), np.str_('escalated'), np.str_('com')]
Custom split vocab: ['', '[UNK]', np.str_('on'), np.str_('west'), np.str_('to'), np.str_('tkt'), np.str_('ticket'), np.str_('support'), np.str_('spiked'), np.str_('sla'), np.str_('payment'), np.str_('order'), np.str_('ops'), np.str_('latency'), np.str_('l2'), np.str_('in'), np.str_('https'), np.str_('example'), np.str_('eu'), np.str_('escalated')]


### C) `output_mode`, `ngrams`, `max_tokens`, `output_sequence_length`

When to use:
- `int`: sequence models (Embedding + RNN/CNN/Transformer)
- `multi_hot`: fast baselines with token presence
- `count`: frequency-sensitive models
- `tf-idf`: ranking/search/classic ML
- `ngrams=(1,2)`: phrase features like `not approved`
- `max_tokens`: memory control and noise reduction
- `output_sequence_length`: fixed serving shape for `int` mode

In [5]:
vec_tfidf = tf.keras.layers.TextVectorization(
    standardize=standardize_aggressive,
    split='whitespace',
    output_mode='tf-idf',
    ngrams=(1, 2),
    max_tokens=50
)
vec_tfidf.adapt(texts)
features = vec_tfidf(texts)

print('TF-IDF vocab size:', len(vec_tfidf.get_vocabulary()))
print('Feature shape:', features.shape)
print('First row sample:', tf.round(features[0] * 1000) / 1000)

TF-IDF vocab size: 35
Feature shape: (3, 35)
First row sample: tf.Tensor(
[0.    0.56  0.56  0.693 0.693 0.693 0.693 0.693 0.916 0.    0.    0.
 0.    0.    0.    0.    0.916 0.916 0.    0.    0.    0.    0.    0.
 0.916 0.916 0.916 0.916 0.    0.    0.916 0.916 0.916 0.    0.   ], shape=(35,), dtype=float32)


## 3) Simple ML Concepts and Keywords

- Token: a text piece after splitting (`payment`, `failed`)
- Vocabulary: known tokens and IDs
- OOV: unknown token not in vocabulary
- OOV buckets: multiple unknown bins, not just one unknown ID
- Embedding: dense learned vector for each token ID
- Bag of Words: token presence/frequency without order
- TF-IDF: weighs important terms higher, common terms lower
- Train-serve skew: preprocessing mismatch between training and production
- Drift: production text distribution changes over time

## 3A) N-grams: Concepts and Terms in Simple Words

`n-gram` means a group of `n` consecutive tokens.

Common terms:
- `unigram` (`n=1`): one token at a time (example: `payment`)
- `bigram` (`n=2`): two-token phrase (example: `payment failed`)
- `trigram` (`n=3`): three-token phrase (example: `card not approved`)
- `ngram range` (example `(1, 2)`): include both unigrams and bigrams

Why n-grams matter:
- Single words may lose phrase meaning.
- Phrase features capture context like `not approved` vs `approved`.
- This often improves search, ranking, and classical ML text models.

In [6]:
ngram_text = tf.constant([
    'card approved quickly',
    'card not approved',
    'payment failed due to timeout'
])

# Unigram model: learns single-word features only
vec_uni = tf.keras.layers.TextVectorization(
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='tf-idf',
    ngrams=1
)
vec_uni.adapt(ngram_text)

# Unigram + bigram model: learns words and two-word phrases
vec_uni_bi = tf.keras.layers.TextVectorization(
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='tf-idf',
    ngrams=(1, 2)
)
vec_uni_bi.adapt(ngram_text)

print('Unigram vocabulary sample:', vec_uni.get_vocabulary()[:20])
print('Uni+Bi vocabulary sample :', vec_uni_bi.get_vocabulary()[:25])

sample = tf.constant(['card not approved'])
print('\nInput sample:', sample.numpy())
print('Unigram feature shape :', vec_uni(sample).shape)
print('Uni+Bi feature shape  :', vec_uni_bi(sample).shape)

print('\nObservation: Uni+Bi can include phrase features such as "not approved".')

Unigram vocabulary sample: ['[UNK]', np.str_('card'), np.str_('approved'), np.str_('to'), np.str_('timeout'), np.str_('quickly'), np.str_('payment'), np.str_('not'), np.str_('failed'), np.str_('due')]
Uni+Bi vocabulary sample : ['[UNK]', np.str_('card'), np.str_('approved'), np.str_('to timeout'), np.str_('to'), np.str_('timeout'), np.str_('quickly'), np.str_('payment failed'), np.str_('payment'), np.str_('not approved'), np.str_('not'), np.str_('failed due'), np.str_('failed'), np.str_('due to'), np.str_('due'), np.str_('card not'), np.str_('card approved'), np.str_('approved quickly')]

Input sample: [b'card not approved']
Unigram feature shape : (1, 10)
Uni+Bi feature shape  : (1, 18)

Observation: Uni+Bi can include phrase features such as "not approved".


### N-gram Use Cases: Where and Why

1. Search and query ranking
- Example query: `refund status delayed`
- Why n-grams: phrases like `refund status` are stronger signals than separate words.

2. Sentiment and intent with negation
- Example text: `not approved`, `not working`
- Why n-grams: bigrams preserve negation meaning. Unigrams may miss this context.

3. Compliance and risk alerts
- Example phrase: `unauthorized access`, `policy violation`
- Why n-grams: phrase-level patterns are closer to business rules.

4. Support ticket triage
- Example phrase: `payment failed`, `login issue`
- Why n-grams: common issue phrases improve routing precision.

5. E-commerce product search and matching
- Example query: `wireless mouse`, `gaming keyboard`
- Why n-grams: product phrases reduce false matches from isolated tokens.

6. Voice/chat command understanding
- Example: `turn off lights`, `set alarm now`
- Why n-grams: command phrases map better to intent than single words.

When to prefer ngrams:
- Use `ngrams=(1, 2)` for most practical setups.
- Add trigrams only if data is large enough and quality gain justifies extra features.
- If model becomes heavy, reduce `max_tokens` or keep only bigrams for target domains.

### Special Edge Scenarios (Important in Production)

1. Negation flipping meaning
- `approved` vs `not approved` can mean opposite outcomes.
- Use bigrams to preserve `not approved`.

2. Word-order sensitivity
- `payment failed` vs `failed payment` may carry different intent depending on domain.
- N-grams help maintain local order information.

3. Very short text inputs
- Inputs like `failed` or `urgent` have little context.
- Keep unigrams enabled (`ngrams=(1,2)`) so short inputs still produce useful features.

4. Noisy punctuation and mixed separators
- `login-issue`, `login/issue`, `login_issue` may represent same intent.
- Normalize text in `standardize` before applying n-grams.

5. Vocabulary explosion with high-order n-grams
- Adding trigrams on small data can create sparse noisy features.
- Control with `max_tokens`, validation metrics, and usually start from `(1,2)`.

In [7]:
edge_cases = tf.constant([
    'approved',
    'not approved',
    'payment failed',
    'failed payment',
    'login-issue',
    'login issue',
    'urgent'
])

edge_vec = tf.keras.layers.TextVectorization(
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='tf-idf',
    ngrams=(1, 2),
    max_tokens=100
)
edge_vec.adapt(edge_cases)

vocab = edge_vec.get_vocabulary()
print('Edge-case vocabulary sample:', vocab[:40])

# Check if critical phrases are captured as explicit n-gram features
critical_terms = ['not approved', 'payment failed', 'failed payment', 'login issue']
for term in critical_terms:
    print(f"Contains '{term}'?", term in vocab)

print('\nInput samples:')
for x in edge_cases.numpy():
    print('-', x)

print('\nFeature tensor shape:', edge_vec(edge_cases).shape)
print('Note: Use this check in production experiments to validate phrase coverage.')

Edge-case vocabulary sample: ['[UNK]', np.str_('payment'), np.str_('failed'), np.str_('approved'), np.str_('urgent'), np.str_('payment failed'), np.str_('not approved'), np.str_('not'), np.str_('loginissue'), np.str_('login issue'), np.str_('login'), np.str_('issue'), np.str_('failed payment')]
Contains 'not approved'? True
Contains 'payment failed'? True
Contains 'failed payment'? True
Contains 'login issue'? True

Input samples:
- b'approved'
- b'not approved'
- b'payment failed'
- b'failed payment'
- b'login-issue'
- b'login issue'
- b'urgent'

Feature tensor shape: (7, 13)
Note: Use this check in production experiments to validate phrase coverage.


### N-grams in Production: What To Do and What To Avoid

What to do:
1. Start with `ngrams=(1, 2)` as default baseline.
- This usually gives strong phrase signal without large feature explosion.

2. Keep text normalization consistent.
- Normalize punctuation/case/separators before n-gram creation.
- Example: map `login-issue`, `login_issue`, `login issue` to a consistent form.

3. Use validation metrics, not assumptions.
- Compare unigram vs (1,2) vs (1,3) on the same split.
- Track precision/recall/F1 or business KPI (search CTR, routing accuracy).

4. Control vocabulary size with `max_tokens`.
- N-grams can grow quickly; cap features to control memory and latency.

5. Monitor production drift and phrase coverage.
- Track OOV/unknown phrase trends and top new phrases in live traffic.

6. Keep train and serving preprocessing identical.
- Prevent train-serve skew by reusing the same TextVectorization config/version.

What to avoid:
1. Do not jump to trigrams on small datasets.
- This often creates sparse noisy features and overfitting.

2. Do not use n-grams without cleanup.
- Raw noisy text creates useless phrase variants and unstable vocab.

3. Do not optimize only for offline accuracy.
- Check memory, latency, and inference cost before shipping.

4. Do not ignore negation phrases.
- Missing bigrams like `not approved` can flip model meaning.

5. Do not keep unlimited vocabulary growth.
- Without caps, models become heavy and harder to maintain.

6. Do not change tokenization rules silently.
- Always version preprocessing and document changes for rollback.

### N-gram Deployment Checklist (Pass/Fail)

Use this before promoting a new n-gram config to production.

| Check | Pass Criteria | Status |
|---|---|---|
| Data split hygiene | `adapt` done on train split only (no leakage) | ☐ |
| Preprocessing parity | Training and serving use same `standardize`/`split` functions | ☐ |
| N-gram scope | Baseline `(1,2)` tested; `(1,3)` only if justified by metrics | ☐ |
| Quality improvement | Validation metric improves over unigram baseline (for example F1/Recall/CTR) | ☐ |
| Negation coverage | Critical phrases like `not approved` represented in vocabulary/features | ☐ |
| Feature size control | `max_tokens` set; feature dimension within infra limits | ☐ |
| Latency budget | Inference latency within SLA under expected load | ☐ |
| Memory budget | Model + vectorizer memory footprint within target | ☐ |
| Drift monitoring | OOV/new phrase monitoring configured in production | ☐ |
| Rollback readiness | Previous tokenizer/vectorizer artifact version available | ☐ |

Go/No-Go rule:
- Go: all mandatory checks pass.
- No-Go: any mandatory check fails or has unknown status.

In [8]:
# Optional: simple go/no-go helper
# Edit each value to True/False based on your run reports.

checks = {
    'data_split_hygiene': True,
    'preprocessing_parity': True,
    'ngram_scope_tested': True,
    'quality_improvement': True,
    'negation_coverage': True,
    'feature_size_control': True,
    'latency_budget': True,
    'memory_budget': True,
    'drift_monitoring': True,
    'rollback_readiness': True,
}

mandatory = [
    'data_split_hygiene',
    'preprocessing_parity',
    'quality_improvement',
    'latency_budget',
    'rollback_readiness',
]

failed_mandatory = [k for k in mandatory if not checks.get(k, False)]
all_mandatory_pass = len(failed_mandatory) == 0

print('Checklist summary:')
for k, v in checks.items():
    print(f"- {k}: {'PASS' if v else 'FAIL'}")

print('\nDecision:', 'GO' if all_mandatory_pass else 'NO-GO')
if failed_mandatory:
    print('Failed mandatory checks:', failed_mandatory)

Checklist summary:
- data_split_hygiene: PASS
- preprocessing_parity: PASS
- ngram_scope_tested: PASS
- quality_improvement: PASS
- negation_coverage: PASS
- feature_size_control: PASS
- latency_budget: PASS
- memory_budget: PASS
- drift_monitoring: PASS
- rollback_readiness: PASS

Decision: GO


## 4) Real-Time Use Cases

### Support Ticket Routing
Input sample: `Ticket TKT-1245 failed after payment timeout on eu-west`
Recommended: custom `standardize` + custom `split` + `output_mode='int'`

### Search Query Ranking
Input sample: `refund card chargeback process`
Recommended: custom `standardize` + `output_mode='tf-idf'` + `ngrams`

### Compliance Alerts
Input sample: `SLA breach and unauthorized access attempt`
Recommended: start with `multi_hot` baseline, scale to `int` sequence model if needed